# 🧠 Research Paper Intelligence Engine (Standalone Colab Version)

This notebook is completely self-contained. Run all cells from top to bottom. It will write the source code to the Colab environment, install dependencies, and launch the Streamlit web app.

In [ ]:
!mkdir -p src
!mkdir -p data
!mkdir -p documents
!mkdir -p embeddings
!mkdir -p vector_store

In [ ]:
%%writefile requirements.txt
# ============================================================
# Research Paper Intelligence Engine — RAG System
# Requirements
# ============================================================

# PDF Processing
PyMuPDF>=1.23.0

# Text Splitting & LLM Chains
langchain>=0.1.0
langchain-community>=0.0.20

# Embeddings
sentence-transformers>=2.2.2

# Vector Store
faiss-cpu>=1.7.4

# HuggingFace Models (Summarization / QA)
transformers>=4.36.0
torch>=2.0.0

# UI
streamlit>=1.30.0

# Utilities
numpy>=1.24.0
tqdm>=4.66.0


In [ ]:
%%writefile config.py
# ============================================================
# config.py — Central Configuration
# All tunable parameters live here so the rest of the code
# never needs magic numbers.
# ============================================================

import os

# ── Paths ────────────────────────────────────────────────────
BASE_DIR        = os.path.dirname(os.path.abspath(__file__))
DATA_DIR        = os.path.join(BASE_DIR, "data")
DOCUMENTS_DIR   = os.path.join(BASE_DIR, "documents")
EMBEDDINGS_DIR  = os.path.join(BASE_DIR, "embeddings")
VECTOR_STORE_DIR= os.path.join(BASE_DIR, "vector_store")

# Create directories if they don't exist
for _dir in [DATA_DIR, DOCUMENTS_DIR, EMBEDDINGS_DIR, VECTOR_STORE_DIR]:
    os.makedirs(_dir, exist_ok=True)

# ── Chunking ─────────────────────────────────────────────────
CHUNK_SIZE      = 500       # characters per chunk
CHUNK_OVERLAP   = 100       # overlap between consecutive chunks

# ── Embedding Model ──────────────────────────────────────────
# A lightweight, high-quality model that runs well on CPU
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ── FAISS ────────────────────────────────────────────────────
FAISS_INDEX_FILE = os.path.join(VECTOR_STORE_DIR, "faiss_index.index")
CHUNKS_META_FILE = os.path.join(VECTOR_STORE_DIR, "chunks_meta.json")

# ── Retrieval ────────────────────────────────────────────────
TOP_K_RESULTS   = 5         # number of chunks to retrieve per query

# ── Summarization / QA Model ────────────────────────────────
# facebook/bart-large-cnn is fine for CPU; swap for a larger
# model if running on GPU/Colab A100.
SUMMARIZATION_MODEL = "facebook/bart-large-cnn"
QA_MODEL            = "google/flan-t5-base"

# ── Logging ──────────────────────────────────────────────────
LOG_LEVEL = "INFO"


In [ ]:
%%writefile src/pdf_processor.py
# ============================================================
# src/pdf_processor.py
# Phase 1 — PDF Processing
#
# Objective:
#   Extract raw text from one or many PDF files, clean it,
#   and save plain-text versions to the documents/ folder.
#
# Architecture:
#   PDFProcessor class
#     ├── extract_text(pdf_path) -> str
#     ├── clean_text(raw_text)   -> str
#     └── process_all(pdf_dir)  -> dict[filename, clean_text]
#
# Dependencies: PyMuPDF (fitz), os, logging, re
# ============================================================

import os
import re
import logging
import fitz  # PyMuPDF

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import DATA_DIR, DOCUMENTS_DIR, LOG_LEVEL

# ── Logger setup ─────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class PDFProcessor:
    """
    Handles extraction and cleaning of text from PDF research papers.

    Usage:
        processor = PDFProcessor()
        texts = processor.process_all()   # reads from data/
        # or process a single file:
        text = processor.extract_text("path/to/paper.pdf")
    """

    def __init__(self, data_dir: str = DATA_DIR, docs_dir: str = DOCUMENTS_DIR):
        self.data_dir = data_dir
        self.docs_dir = docs_dir
        os.makedirs(self.docs_dir, exist_ok=True)
        logger.info("PDFProcessor initialised.")
        logger.info(f"  Input  dir : {self.data_dir}")
        logger.info(f"  Output dir : {self.docs_dir}")

    # ── Core extraction ──────────────────────────────────────

    def extract_text(self, pdf_path: str = None, stream: bytes = None) -> str:
        """
        Extract raw text from a single PDF file page by page.

        Args:
            pdf_path: Absolute or relative path to the PDF (optional).
            stream: Raw bytes of the PDF file (optional).

        Returns:
            A single string containing all pages concatenated.
        """
        if stream is not None:
            logger.info("Extracting text from memory stream")
            try:
                doc = fitz.open(stream=stream, filetype="pdf")
            except Exception as exc:
                raise RuntimeError(f"Failed to open PDF from stream: {exc}") from exc
        elif pdf_path is not None:
            if not os.path.isfile(pdf_path):
                raise FileNotFoundError(f"PDF not found: {pdf_path}")
            logger.info(f"Extracting text from: {os.path.basename(pdf_path)}")
            try:
                doc = fitz.open(pdf_path)
            except Exception as exc:
                raise RuntimeError(f"Failed to open PDF '{pdf_path}': {exc}") from exc
        else:
            raise ValueError("Must provide either pdf_path or stream")

        pages_text = []
        try:
            for page in doc:
                pages_text.append(page.get_text("text"))
            doc.close()
        except Exception as exc:
            raise RuntimeError(f"Error during extraction: {exc}") from exc

        raw_text = "\n".join(pages_text)
        logger.info(f"  Extracted {len(raw_text):,} chars.")
        return raw_text

    # ── Text cleaning ────────────────────────────────────────

    def clean_text(self, raw_text: str) -> str:
        """
        Remove artefacts common in PDF-extracted text:
          - Excessive whitespace / blank lines
          - Hyphenated line breaks (re-join words)
          - Non-ASCII junk characters
          - Page headers/footers patterns

        Args:
            raw_text: The raw string from extract_text().

        Returns:
            A cleaner, more readable string.
        """
        text = raw_text

        # Re-join hyphenated words split across lines (common in PDFs)
        text = re.sub(r"-\n", "", text)

        # Remove excessive newlines (3+ → 2)
        text = re.sub(r"\n{3,}", "\n\n", text)

        # Replace non-breaking spaces and tabs with regular space
        text = text.replace("\xa0", " ").replace("\t", " ")

        # Remove lines that are just page numbers (e.g. "— 3 —" or just "3")
        text = re.sub(r"^\s*[\-–—]?\s*\d+\s*[\-–—]?\s*$", "", text, flags=re.MULTILINE)

        # Collapse multiple spaces into one
        text = re.sub(r" {2,}", " ", text)

        # Strip leading/trailing whitespace from each line
        lines = [line.strip() for line in text.splitlines()]
        text = "\n".join(lines)

        # Final strip
        text = text.strip()

        logger.debug(f"  Cleaned text length: {len(text):,} characters.")
        return text

    # ── Save to disk ─────────────────────────────────────────

    def save_text(self, filename: str, text: str) -> str:
        """
        Save cleaned text to documents/ folder as a .txt file.

        Args:
            filename: Base name (e.g. 'paper1.pdf' → 'paper1.txt').
            text: The cleaned text string.

        Returns:
            Path to the saved .txt file.
        """
        base = os.path.splitext(filename)[0]
        out_path = os.path.join(self.docs_dir, f"{base}.txt")
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(text)
        logger.info(f"  Saved cleaned text → {out_path}")
        return out_path

    # ── Batch processing ─────────────────────────────────────

    def process_all(self) -> dict:
        """
        Process every PDF in data_dir.

        Returns:
            {
                "paper1.pdf": {
                    "raw_text": "...",
                    "clean_text": "...",
                    "txt_path": "documents/paper1.txt"
                },
                ...
            }
        """
        pdf_files = [
            f for f in os.listdir(self.data_dir)
            if f.lower().endswith(".pdf")
        ]

        if not pdf_files:
            logger.warning(f"No PDF files found in {self.data_dir}")
            return {}

        logger.info(f"Found {len(pdf_files)} PDF(s) to process.")
        results = {}

        for pdf_file in pdf_files:
            pdf_path = os.path.join(self.data_dir, pdf_file)
            try:
                raw   = self.extract_text(pdf_path)
                clean = self.clean_text(raw)
                path  = self.save_text(pdf_file, clean)
                results[pdf_file] = {
                    "raw_text"  : raw,
                    "clean_text": clean,
                    "txt_path"  : path,
                }
            except (FileNotFoundError, RuntimeError) as exc:
                logger.error(f"Skipping '{pdf_file}': {exc}")

        logger.info(f"Processing complete. {len(results)}/{len(pdf_files)} PDFs succeeded.")
        return results

    # ── Single-file convenience method ───────────────────────

    def process_single(self, pdf_path: str = None, stream: bytes = None, filename: str = None) -> dict:
        """
        Extract, clean, save and return result for one PDF.

        Args:
            pdf_path: Path to the PDF file (optional).
            stream: Raw bytes of the PDF file (optional).
            filename: Override the filename (optional).

        Returns:
            Dict with keys: raw_text, clean_text, txt_path.
        """
        if filename is None:
            filename = os.path.basename(pdf_path) if pdf_path else "document.pdf"
            
        raw_text  = self.extract_text(pdf_path=pdf_path, stream=stream)
        clean     = self.clean_text(raw_text)
        txt_path  = self.save_text(filename, clean)
        return {
            "raw_text"  : raw_text,
            "clean_text": clean,
            "txt_path"  : txt_path,
        }


# ── Quick test (run this file directly) ─────────────────────
if __name__ == "__main__":
    import sys
    if len(sys.argv) > 1:
        pdf = sys.argv[1]
        processor = PDFProcessor()
        result = processor.process_single(pdf)
        print(f"\n✅ Extracted {len(result['clean_text']):,} characters.")
        print("First 500 chars:\n", result["clean_text"][:500])
    else:
        processor = PDFProcessor()
        results = processor.process_all()
        print(f"\n✅ Processed {len(results)} PDF(s).")


In [ ]:
%%writefile src/chunking.py
# ============================================================
# src/chunking.py
# Phase 2 — Text Chunking
#
# Objective:
#   Split long cleaned text into smaller, overlapping chunks
#   suitable for embedding and retrieval.
#
# Architecture:
#   TextChunker class
#     └── chunk_text(text, metadata) -> list[dict]
#     └── chunk_documents(docs_dict) -> list[dict]
#
# Why chunking matters:
#   Embedding models have a token limit (~512 tokens for
#   MiniLM). Splitting with overlap ensures context is not
#   lost at chunk boundaries.
#
# Dependencies: langchain, logging
# ============================================================

import logging
import os

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import CHUNK_SIZE, CHUNK_OVERLAP, LOG_LEVEL

from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class TextChunker:
    """
    Splits research paper text into overlapping chunks.

    Each chunk is a dict:
      {
          "chunk_id"   : int,           # global unique index
          "source"     : str,           # source PDF filename
          "text"       : str,           # the chunk content
          "char_start" : int,           # approx start position
      }

    Usage:
        chunker = TextChunker()
        chunks = chunker.chunk_text(clean_text, source="paper.pdf")
    """

    def __init__(
        self,
        chunk_size: int = CHUNK_SIZE,
        chunk_overlap: int = CHUNK_OVERLAP
    ):
        self.chunk_size    = chunk_size
        self.chunk_overlap = chunk_overlap

        # RecursiveCharacterTextSplitter tries to split on
        # paragraph → sentence → word boundaries in order,
        # falling back to characters only as a last resort.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.chunk_size,
            chunk_overlap=self.chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
            length_function=len,
        )
        logger.info(
            f"TextChunker ready | chunk_size={chunk_size} | overlap={chunk_overlap}"
        )

    # ── Single document ──────────────────────────────────────

    def chunk_text(self, text: str, source: str = "unknown") -> list:
        """
        Chunk a single document's text.

        Args:
            text   : Cleaned text string.
            source : Filename/label for provenance tracking.

        Returns:
            List of chunk dicts (see class docstring).
        """
        if not text or not text.strip():
            logger.warning(f"Empty text provided for source: {source}")
            return []

        raw_chunks = self.splitter.split_text(text)

        chunks = []
        for idx, chunk_text in enumerate(raw_chunks):
            chunks.append({
                "chunk_id"  : idx,
                "source"    : source,
                "text"      : chunk_text.strip(),
                "char_start": text.find(chunk_text[:50]) if len(chunk_text) >= 50 else 0,
            })

        logger.info(
            f"'{source}' → {len(chunks)} chunks "
            f"(avg {sum(len(c['text']) for c in chunks) // max(len(chunks),1)} chars each)"
        )
        return chunks

    # ── Multiple documents ───────────────────────────────────

    def chunk_documents(self, docs_dict: dict) -> list:
        """
        Chunk multiple documents and return a flat list with
        globally unique chunk IDs.

        Args:
            docs_dict: Output of PDFProcessor.process_all()
                       {filename: {"clean_text": str, ...}}

        Returns:
            Flat list of all chunks across all documents,
            with globally sequential chunk_ids.
        """
        all_chunks = []
        global_id  = 0

        for filename, doc_data in docs_dict.items():
            clean_text = doc_data.get("clean_text", "")
            chunks     = self.chunk_text(clean_text, source=filename)

            # Re-number chunk_ids globally
            for chunk in chunks:
                chunk["chunk_id"] = global_id
                global_id += 1

            all_chunks.extend(chunks)

        logger.info(f"Total chunks across all documents: {len(all_chunks)}")
        return all_chunks

    # ── Utility ──────────────────────────────────────────────

    def chunk_from_file(self, txt_path: str) -> list:
        """
        Load a saved .txt file and chunk it.

        Args:
            txt_path: Path to a plain-text file.

        Returns:
            List of chunk dicts.
        """
        if not os.path.isfile(txt_path):
            raise FileNotFoundError(f"Text file not found: {txt_path}")

        source = os.path.basename(txt_path)
        with open(txt_path, "r", encoding="utf-8") as f:
            text = f.read()

        return self.chunk_text(text, source=source)


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample = """
    Attention Is All You Need

    Abstract
    The dominant sequence transduction models are based on complex recurrent or
    convolutional neural networks that include an encoder and a decoder.
    The best performing models also connect the encoder and decoder through an
    attention mechanism. We propose a new simple network architecture, the
    Transformer, based solely on attention mechanisms, dispensing with recurrence
    and convolutions entirely.

    1. Introduction
    Recurrent neural networks, long short-term memory and gated recurrent neural
    networks in particular, have been firmly established as state of the art
    approaches in sequence modelling and transduction problems such as language
    modelling and machine translation.
    """ * 10  # repeat to create enough text for chunking

    chunker = TextChunker(chunk_size=200, chunk_overlap=40)
    chunks  = chunker.chunk_text(sample, source="test_paper.pdf")

    print(f"\n✅ Created {len(chunks)} chunks.\n")
    for c in chunks[:3]:
        print(f"  Chunk {c['chunk_id']} | {c['source']} | {len(c['text'])} chars")
        print(f"  Preview: {c['text'][:80]}...\n")


In [ ]:
%%writefile src/embeddings.py
# ============================================================
# src/embeddings.py
# Phase 3 — Embedding Generation
#
# Objective:
#   Convert text chunks into dense vector representations
#   (embeddings) using a local Sentence Transformer model.
#
# Architecture:
#   EmbeddingGenerator class
#     ├── embed_chunks(chunks)       -> np.ndarray  (N, D)
#     ├── embed_query(query)         -> np.ndarray  (D,)
#     ├── save_embeddings(arr, path)
#     └── load_embeddings(path)      -> np.ndarray
#
# Model: all-MiniLM-L6-v2
#   - 384-dimensional embeddings
#   - Fast on CPU, excellent quality for semantic search
#   - Downloads automatically from HuggingFace on first use
#
# Dependencies: sentence-transformers, numpy, tqdm, logging
# ============================================================

import os
import logging

import numpy as np
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import EMBEDDING_MODEL, EMBEDDINGS_DIR, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class EmbeddingGenerator:
    """
    Generates sentence embeddings for text chunks and queries.

    Usage:
        gen = EmbeddingGenerator()
        embeddings = gen.embed_chunks(chunks)   # shape: (N, 384)
        query_vec  = gen.embed_query("What is attention?")
    """

    def __init__(self, model_name: str = EMBEDDING_MODEL):
        logger.info(f"Loading embedding model: {model_name}")
        logger.info("  (First run downloads ~90 MB from HuggingFace — once only)")
        self.model_name = model_name
        self.model      = SentenceTransformer(model_name)
        self.dim        = self.model.get_sentence_embedding_dimension()
        logger.info(f"  Model ready | embedding dim = {self.dim}")

    # ── Embed chunks ─────────────────────────────────────────

    def embed_chunks(self, chunks: list, batch_size: int = 32) -> np.ndarray:
        """
        Generate embeddings for a list of chunk dicts.

        Args:
            chunks    : List of chunk dicts with key 'text'.
            batch_size: How many texts to encode at once.

        Returns:
            numpy array of shape (N, embedding_dim).

        Raises:
            ValueError: If chunks list is empty.
        """
        if not chunks:
            raise ValueError("chunks list is empty — nothing to embed.")

        texts = [c["text"] for c in chunks]
        logger.info(f"Embedding {len(texts)} chunks (batch_size={batch_size}) …")

        embeddings = self.model.encode(
            texts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            normalize_embeddings=True,  # L2-normalise for cosine similarity
        )

        logger.info(f"Embeddings shape: {embeddings.shape}")
        return embeddings

    # ── Embed single query ───────────────────────────────────

    def embed_query(self, query: str) -> np.ndarray:
        """
        Embed a single query string.

        Args:
            query: The user question or search string.

        Returns:
            1-D numpy array of shape (embedding_dim,).
        """
        if not query or not query.strip():
            raise ValueError("Query string is empty.")

        vec = self.model.encode(
            [query],
            convert_to_numpy=True,
            normalize_embeddings=True,
        )[0]

        logger.debug(f"Query embedded | shape: {vec.shape}")
        return vec

    # ── Persistence ──────────────────────────────────────────

    def save_embeddings(self, embeddings: np.ndarray, name: str = "embeddings") -> str:
        """
        Save embeddings to disk as a .npy file.

        Args:
            embeddings: numpy array to save.
            name      : Base filename (without extension).

        Returns:
            Path to the saved file.
        """
        os.makedirs(EMBEDDINGS_DIR, exist_ok=True)
        path = os.path.join(EMBEDDINGS_DIR, f"{name}.npy")
        np.save(path, embeddings)
        logger.info(f"Embeddings saved → {path}  ({embeddings.shape})")
        return path

    def load_embeddings(self, name: str = "embeddings") -> np.ndarray:
        """
        Load embeddings from disk.

        Args:
            name: Base filename (without .npy extension).

        Returns:
            numpy array of embeddings.

        Raises:
            FileNotFoundError: If the .npy file does not exist.
        """
        path = os.path.join(EMBEDDINGS_DIR, f"{name}.npy")
        if not os.path.isfile(path):
            raise FileNotFoundError(f"Embeddings file not found: {path}")
        embeddings = np.load(path)
        logger.info(f"Embeddings loaded ← {path}  ({embeddings.shape})")
        return embeddings


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample_chunks = [
        {"chunk_id": 0, "source": "test.pdf", "text": "Attention is all you need."},
        {"chunk_id": 1, "source": "test.pdf", "text": "Transformers changed NLP forever."},
        {"chunk_id": 2, "source": "test.pdf", "text": "BERT uses bidirectional attention."},
    ]

    gen = EmbeddingGenerator()
    emb = gen.embed_chunks(sample_chunks)
    print(f"\n✅ Chunk embeddings shape: {emb.shape}")

    qvec = gen.embed_query("What is the transformer model?")
    print(f"✅ Query embedding shape : {qvec.shape}")

    # Test cosine similarity (arrays are already normalised)
    scores = emb @ qvec
    for i, s in enumerate(scores):
        print(f"  Chunk {i} similarity: {s:.4f}")


In [ ]:
%%writefile src/vector_db.py
# ============================================================
# src/vector_db.py
# Phase 4 — FAISS Vector Store
#
# Objective:
#   Build a FAISS index from embeddings, persist it to disk,
#   reload it, and perform fast nearest-neighbour search.
#
# Architecture:
#   VectorDB class
#     ├── build_index(embeddings)
#     ├── save(index_path, meta_path)
#     ├── load(index_path, meta_path)
#     ├── search(query_vec, top_k) -> list[dict]
#     └── add_embeddings(new_embeddings, new_chunks)
#
# Why FAISS?
#   FAISS (Facebook AI Similarity Search) provides highly
#   optimised C++ similarity search accessible from Python.
#   IndexFlatIP uses inner-product (= cosine when vectors
#   are L2-normalised) — perfect for our use case.
#
# Dependencies: faiss-cpu, numpy, json, logging
# ============================================================

import os
import json
import logging

import numpy as np
import faiss

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import FAISS_INDEX_FILE, CHUNKS_META_FILE, TOP_K_RESULTS, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class VectorDB:
    """
    Wraps a FAISS index alongside chunk metadata for
    building, saving, loading, and querying a vector store.

    Usage:
        db = VectorDB()
        db.build_index(embeddings, chunks)
        db.save()
        # --- later ---
        db = VectorDB()
        db.load()
        results = db.search(query_vec, top_k=5)
    """

    def __init__(
        self,
        index_path: str = FAISS_INDEX_FILE,
        meta_path : str = CHUNKS_META_FILE,
    ):
        self.index_path = index_path
        self.meta_path  = meta_path
        self.index      = None   # faiss.Index instance
        self.chunks     = []     # list of chunk dicts (metadata)
        logger.info("VectorDB initialised.")

    # ── Build ────────────────────────────────────────────────

    def build_index(self, embeddings: np.ndarray, chunks: list) -> None:
        """
        Create a new FAISS index from embeddings.

        Args:
            embeddings: numpy array of shape (N, D), L2-normalised.
            chunks    : List of chunk dicts matching the embeddings.

        Raises:
            ValueError: If embeddings is empty or shapes mismatch.
        """
        if embeddings is None or len(embeddings) == 0:
            raise ValueError("Cannot build index from empty embeddings.")
        if len(embeddings) != len(chunks):
            raise ValueError(
                f"Mismatch: {len(embeddings)} embeddings vs {len(chunks)} chunks."
            )

        dim = embeddings.shape[1]
        logger.info(f"Building FAISS IndexFlatIP | dim={dim} | vectors={len(embeddings)}")

        # IndexFlatIP = exact inner-product search.
        # With L2-normalised vectors, IP == cosine similarity.
        self.index  = faiss.IndexFlatIP(dim)
        self.chunks = chunks

        # FAISS requires float32
        embeddings_f32 = embeddings.astype(np.float32)
        self.index.add(embeddings_f32)

        logger.info(f"Index built. Total vectors stored: {self.index.ntotal}")

    # ── Save ─────────────────────────────────────────────────

    def save(
        self,
        index_path: str | None = None,
        meta_path : str | None = None,
    ) -> None:
        """
        Persist the FAISS index and chunk metadata to disk.

        Args:
            index_path: Override default index file path.
            meta_path : Override default metadata file path.

        Raises:
            RuntimeError: If the index has not been built yet.
        """
        if self.index is None:
            raise RuntimeError("No index to save. Call build_index() first.")

        idx_path  = index_path or self.index_path
        meta_path = meta_path  or self.meta_path

        os.makedirs(os.path.dirname(idx_path),  exist_ok=True)
        os.makedirs(os.path.dirname(meta_path), exist_ok=True)

        faiss.write_index(self.index, idx_path)
        logger.info(f"FAISS index saved → {idx_path}")

        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(self.chunks, f, ensure_ascii=False, indent=2)
        logger.info(f"Chunk metadata saved → {meta_path}")

    # ── Load ─────────────────────────────────────────────────

    def load(
        self,
        index_path: str | None = None,
        meta_path : str | None = None,
    ) -> None:
        """
        Load FAISS index and chunk metadata from disk.

        Raises:
            FileNotFoundError: If either file does not exist.
        """
        idx_path  = index_path or self.index_path
        meta_path = meta_path  or self.meta_path

        if not os.path.isfile(idx_path):
            raise FileNotFoundError(f"FAISS index not found: {idx_path}")
        if not os.path.isfile(meta_path):
            raise FileNotFoundError(f"Chunks metadata not found: {meta_path}")

        self.index = faiss.read_index(idx_path)
        logger.info(f"FAISS index loaded ← {idx_path}  ({self.index.ntotal} vectors)")

        with open(meta_path, "r", encoding="utf-8") as f:
            self.chunks = json.load(f)
        logger.info(f"Chunk metadata loaded ← {meta_path}  ({len(self.chunks)} chunks)")

    # ── Search ───────────────────────────────────────────────

    def search(self, query_vec: np.ndarray, top_k: int = TOP_K_RESULTS) -> list:
        """
        Find the top-k most similar chunks to a query vector.

        Args:
            query_vec: 1-D numpy array (embedding_dim,), L2-normalised.
            top_k    : Number of results to return.

        Returns:
            List of result dicts:
            [
              {
                "rank"   : 1,
                "score"  : 0.92,        # cosine similarity
                "chunk_id": 42,
                "source" : "paper.pdf",
                "text"   : "...",
              },
              ...
            ]

        Raises:
            RuntimeError: If index is not loaded.
        """
        if self.index is None:
            raise RuntimeError("Index not loaded. Call build_index() or load() first.")

        query_f32 = query_vec.astype(np.float32).reshape(1, -1)
        scores, indices = self.index.search(query_f32, top_k)

        results = []
        for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), start=1):
            if idx == -1:               # FAISS returns -1 for missing results
                continue
            chunk = self.chunks[idx].copy()
            chunk["rank"]  = rank
            chunk["score"] = float(score)
            results.append(chunk)

        logger.debug(f"Search returned {len(results)} results.")
        return results

    # ── Incremental add ──────────────────────────────────────

    def add_embeddings(self, new_embeddings: np.ndarray, new_chunks: list) -> None:
        """
        Add more embeddings to an existing index (incremental update).

        Args:
            new_embeddings: numpy array (M, D).
            new_chunks    : Corresponding chunk dicts (length M).
        """
        if self.index is None:
            raise RuntimeError("Index not initialised. Call build_index() first.")

        self.index.add(new_embeddings.astype(np.float32))
        self.chunks.extend(new_chunks)
        logger.info(
            f"Added {len(new_chunks)} chunks. Total: {self.index.ntotal} vectors."
        )

    # ── Utility ──────────────────────────────────────────────

    @property
    def is_ready(self) -> bool:
        """True if the index is built/loaded and non-empty."""
        return self.index is not None and self.index.ntotal > 0

    def get_stats(self) -> dict:
        """Return basic statistics about the current index."""
        return {
            "total_vectors": self.index.ntotal if self.index else 0,
            "total_chunks" : len(self.chunks),
            "sources"      : list({c.get("source", "?") for c in self.chunks}),
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    import sys
    sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
    from src.embeddings import EmbeddingGenerator

    chunks = [
        {"chunk_id": 0, "source": "test.pdf", "text": "Attention is all you need."},
        {"chunk_id": 1, "source": "test.pdf", "text": "Transformers use multi-head attention."},
        {"chunk_id": 2, "source": "test.pdf", "text": "BERT fine-tunes on downstream tasks."},
    ]

    gen = EmbeddingGenerator()
    emb = gen.embed_chunks(chunks)

    db = VectorDB()
    db.build_index(emb, chunks)
    db.save()

    print("\n✅ Index built and saved.")
    print(f"   Stats: {db.get_stats()}")

    # Reload and search
    db2 = VectorDB()
    db2.load()
    qvec = gen.embed_query("What model uses self-attention?")
    results = db2.search(qvec, top_k=2)

    print("\n🔍 Search results:")
    for r in results:
        print(f"  Rank {r['rank']} | Score {r['score']:.4f} | {r['source']}")
        print(f"  Text: {r['text']}\n")


In [ ]:
%%writefile src/retriever.py
# ============================================================
# src/retriever.py
# Phase 5 — Semantic Search / Retriever
#
# Objective:
#   Provide a clean, high-level API that takes a natural-
#   language question and returns the most relevant text
#   chunks from the FAISS index.
#
# Architecture:
#   Retriever class
#     └── retrieve(query, top_k) -> list[dict]
#     └── retrieve_with_context(query, top_k) -> str (joined)
#
# This module is the "R" in RAG — it bridges the query
# and the vector store, formatting results for the LLM.
#
# Dependencies: embeddings.py, vector_db.py
# ============================================================

import os
import logging

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import TOP_K_RESULTS, LOG_LEVEL
from src.embeddings import EmbeddingGenerator
from src.vector_db  import VectorDB

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class Retriever:
    """
    Semantic search over the FAISS index.

    Usage:
        retriever = Retriever()
        retriever.load_index()
        results = retriever.retrieve("What are the key findings?")
        context = retriever.retrieve_with_context("What is BERT?")
    """

    def __init__(
        self,
        embedding_gen: EmbeddingGenerator | None = None,
        vector_db    : VectorDB | None = None,
    ):
        """
        Args:
            embedding_gen: Pre-initialised EmbeddingGenerator.
                           If None, a new one is created automatically.
            vector_db    : Pre-initialised VectorDB.
                           If None, a new one is created; call load_index().
        """
        self.embedding_gen = embedding_gen or EmbeddingGenerator()
        self.vector_db     = vector_db     or VectorDB()
        logger.info("Retriever initialised.")

    # ── Index management ─────────────────────────────────────

    def load_index(self) -> None:
        """Load the FAISS index and chunk metadata from disk."""
        self.vector_db.load()
        logger.info("Retriever: index loaded and ready.")

    def is_ready(self) -> bool:
        """Return True if the index is loaded and non-empty."""
        return self.vector_db.is_ready

    # ── Core retrieval ───────────────────────────────────────

    def retrieve(self, query: str, top_k: int = TOP_K_RESULTS) -> list:
        """
        Embed a query and return the top-k matching chunks.

        Args:
            query  : Natural language question or search string.
            top_k  : Number of results to return.

        Returns:
            List of chunk dicts, each with a 'score' field.

        Raises:
            RuntimeError: If the index is not ready.
            ValueError  : If query is empty.
        """
        if not query or not query.strip():
            raise ValueError("Query must not be empty.")
        if not self.is_ready():
            raise RuntimeError(
                "Vector index is not ready. Call load_index() or build the index first."
            )

        logger.info(f"Retrieving top-{top_k} chunks for query: '{query[:60]}...' ")
        query_vec = self.embedding_gen.embed_query(query)
        results   = self.vector_db.search(query_vec, top_k=top_k)

        logger.info(f"  Retrieved {len(results)} chunks.")
        for r in results:
            logger.debug(
                f"  Rank {r['rank']} | score={r['score']:.4f} | source={r['source']}"
            )

        return results

    # ── Context string for LLM prompt ────────────────────────

    def retrieve_with_context(
        self,
        query : str,
        top_k : int = TOP_K_RESULTS,
        sep   : str = "\n\n---\n\n",
    ) -> str:
        """
        Retrieve top-k chunks and join them into a single
        context string for use in an LLM prompt.

        Args:
            query: The user's question.
            top_k: Number of chunks to include.
            sep  : Separator between chunks.

        Returns:
            A multi-paragraph string with source annotations.
        """
        results  = self.retrieve(query, top_k=top_k)
        parts    = []

        for r in results:
            header = f"[Source: {r['source']} | Rank: {r['rank']} | Score: {r['score']:.3f}]"
            parts.append(f"{header}\n{r['text']}")

        context = sep.join(parts)
        logger.debug(f"Context length: {len(context):,} characters")
        return context

    # ── Source-filtered retrieval ─────────────────────────────

    def retrieve_from_source(
        self,
        query : str,
        source: str,
        top_k : int = TOP_K_RESULTS,
    ) -> list:
        """
        Retrieve chunks restricted to a specific source document.

        Args:
            query : The search query.
            source: Filename of the paper to restrict to.
            top_k : Number of results to return.

        Returns:
            Filtered list of chunk dicts.
        """
        all_results = self.retrieve(query, top_k=top_k * 3)  # over-fetch then filter
        filtered = [r for r in all_results if r.get("source") == source]
        return filtered[:top_k]


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    # This test assumes vector_db.py was run first to build the index
    retriever = Retriever()

    if not retriever.is_ready():
        print("⚠️  Index not found — run vector_db.py first to build the index.")
    else:
        results = retriever.retrieve("What is the attention mechanism?", top_k=3)
        print(f"\n✅ Retrieved {len(results)} results.\n")
        for r in results:
            print(f"  Rank {r['rank']} | Score {r['score']:.4f} | {r['source']}")
            print(f"  {r['text'][:120]}…\n")


In [ ]:
%%writefile src/rag_pipeline.py
# ============================================================
# src/rag_pipeline.py
# Phase 6 — Retrieval-Augmented Generation (RAG) Pipeline
#
# Objective:
#   Answer user questions by:
#     1. Retrieving relevant context chunks (Retriever)
#     2. Building a prompt with context + question
#     3. Running a local HuggingFace QA model for the answer
#
# Architecture:
#   RAGPipeline class
#     ├── answer(question, top_k) -> dict
#     │     returns: answer, context, sources, confidence
#     └── _build_prompt(question, context) -> str
#
# Model: deepset/roberta-base-squad2
#   - Extractive QA: finds the answer span within the context
#   - Works fully offline after first download
#   - Fast on CPU
#
# For generative answers (replacing extractive QA with a
# seq2seq model), swap to google/flan-t5-base and use the
# text2text-generation pipeline.
#
# Dependencies: transformers, retriever.py
# ============================================================

import os
import logging

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import QA_MODEL, TOP_K_RESULTS, LOG_LEVEL
from src.retriever import Retriever

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)


class RAGPipeline:
    """
    End-to-end Retrieval-Augmented Generation pipeline.

    Combine semantic retrieval with extractive QA to answer
    natural language questions from your research papers.

    Usage:
        rag = RAGPipeline()
        result = rag.answer("What dataset was used for evaluation?")
        print(result["answer"])
        print(result["sources"])
    """

    def __init__(
        self,
        retriever : Retriever | None = None,
        qa_model  : str = QA_MODEL,
    ):
        """
        Args:
            retriever: Pre-initialised Retriever.
                       If None, a new one is created; index auto-loaded.
            qa_model : HuggingFace model ID for QA.
        """
        # ── Retriever setup ──
        if retriever is not None:
            self.retriever = retriever
        else:
            self.retriever = Retriever()
            # Try to load the index; warn if not available yet
            try:
                self.retriever.load_index()
            except FileNotFoundError:
                logger.warning(
                    "No vector index found. "
                    "Call build_index_from_pdfs() or ingest_pdfs() first."
                )

        # ── QA model setup ──
        logger.info(f"Loading QA model: {qa_model}")
        logger.info("  (First run downloads the model from HuggingFace — once only)")
        self.qa_model_name = qa_model
        self.tokenizer = AutoTokenizer.from_pretrained(qa_model)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(qa_model)
        logger.info("RAGPipeline ready.")

    # ── Core QA ──────────────────────────────────────────────

    def answer(self, question: str, top_k: int = TOP_K_RESULTS) -> dict:
        """
        Answer a question using retrieved context.

        Args:
            question: The user's natural language question.
            top_k   : Number of context chunks to retrieve.

        Returns:
            {
                "question"  : str,
                "answer"    : str,   # extracted answer span
                "confidence": float, # model confidence (0–1)
                "context"   : str,   # full context fed to QA model
                "sources"   : list,  # [{"source": ..., "score": ...}]
            }

        Raises:
            RuntimeError: If the retriever index is not ready.
            ValueError  : If the question is empty.
        """
        if not question or not question.strip():
            raise ValueError("Question must not be empty.")

        if not self.retriever.is_ready():
            raise RuntimeError(
                "No vector index found. Please upload and index papers first."
            )

        # Step 1: Retrieve relevant chunks
        logger.info(f"RAG → Question: '{question[:80]}'")
        chunks = self.retriever.retrieve(question, top_k=top_k)

        # Step 2: Build context from retrieved chunks
        context_parts  = [c["text"] for c in chunks]
        context        = " ".join(context_parts)  # flat string for QA model
        sources        = [{"source": c["source"], "score": c["score"]} for c in chunks]

        # Step 3: Run Generative QA model
        logger.info(f"Running QA model on context ({len(context)} chars) …")
        try:
            prompt = f"Answer the following question based only on the provided context. If the context does not contain the answer, say 'I cannot answer this based on the provided text'.\n\nContext:\n{context}\n\nQuestion:\n{question}\n\nAnswer:"
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=150,
                    num_beams=4,
                    early_stopping=True
                )
            answer = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            confidence = 1.0
        except Exception as exc:
            logger.error(f"QA model failed: {exc}")
            answer     = "Could not generate an answer. Please rephrase your question."
            confidence = 0.0

        logger.info(f"Answer: '{answer[:80]}'")

        return {
            "question"  : question,
            "answer"    : answer,
            "confidence": confidence,
            "context"   : context,
            "sources"   : sources,
        }

    # ── Full ingest pipeline (convenience) ───────────────────

    def ingest_and_build_index(
        self,
        pdf_paths: list,
        force_rebuild: bool = False,
    ) -> dict:
        """
        Full pipeline: extract PDFs → chunk → embed → build FAISS index.
        Call this once when new PDFs are uploaded.

        Args:
            pdf_paths    : List of paths to PDF files.
            force_rebuild: If True, rebuild even if index exists.

        Returns:
            {
                "num_pdfs"  : int,
                "num_chunks": int,
                "index_ready": bool,
            }
        """
        import shutil
        from src.pdf_processor import PDFProcessor
        from src.chunking      import TextChunker
        from src.embeddings    import EmbeddingGenerator
        from src.vector_db     import VectorDB
        from config            import DATA_DIR

        logger.info(f"Ingesting {len(pdf_paths)} PDF(s) …")

        # Copy PDFs to data dir
        os.makedirs(DATA_DIR, exist_ok=True)
        for p in pdf_paths:
            dest = os.path.join(DATA_DIR, os.path.basename(p))
            if not os.path.isfile(dest) or force_rebuild:
                shutil.copy2(p, dest)

        # Phase 1 — Extract
        processor = PDFProcessor()
        docs      = {}
        for p in pdf_paths:
            filename = os.path.basename(p)
            result   = processor.process_single(p)
            docs[filename] = result

        # Phase 2 — Chunk
        chunker    = TextChunker()
        all_chunks = chunker.chunk_documents(docs)

        # Phase 3 — Embed
        gen        = EmbeddingGenerator()
        embeddings = gen.embed_chunks(all_chunks)
        gen.save_embeddings(embeddings)

        # Phase 4 — Index
        db = VectorDB()
        db.build_index(embeddings, all_chunks)
        db.save()

        # Update retriever's reference
        self.retriever.vector_db = db
        logger.info("Ingest complete. Index ready.")

        return {
            "num_pdfs"   : len(pdf_paths),
            "num_chunks" : len(all_chunks),
            "index_ready": True,
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    rag = RAGPipeline()
    if rag.retriever.is_ready():
        result = rag.answer("What is the main contribution of this paper?")
        print(f"\n✅ Answer     : {result['answer']}")
        print(f"   Confidence : {result['confidence']:.3f}")
        print(f"   Sources    : {[s['source'] for s in result['sources']]}")
    else:
        print("⚠️  No index found. Please ingest PDFs first.")


In [ ]:
%%writefile src/summarizer.py
# ============================================================
# src/summarizer.py
# Phase 7 & 8 — Summarization + Research Insights Extraction
#
# Objective:
#   1. Generate a concise abstract-style summary of a paper.
#   2. Extract structured research insights:
#      - Key Findings
#      - Limitations
#      - Future Work
#
# Architecture:
#   Summarizer class
#     ├── summarize(text)               -> str
#     ├── extract_insights(text)        -> dict
#     └── full_analysis(chunks, source) -> dict
#
# Strategy:
#   - Use BART (facebook/bart-large-cnn) for summarization.
#   - Use keyword-pattern extraction as a reliable, fast way
#     to pull structured sections without needing a GPU.
#   - Fallback: retrieval-based extraction when sections
#     are not explicitly labelled.
#
# Dependencies: transformers, retriever.py (optional)
# ============================================================

import re
import os
import logging

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

import sys
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from config import SUMMARIZATION_MODEL, LOG_LEVEL

# ── Logger ───────────────────────────────────────────────────
logging.basicConfig(
    level=getattr(logging, LOG_LEVEL),
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s"
)
logger = logging.getLogger(__name__)

# ── Section header patterns (case-insensitive) ────────────────
_FINDINGS_PATTERNS  = [
    r"(?i)(key\s+(?:findings?|contributions?|results?|insights?|discoveries))",
    r"(?i)(main\s+(?:findings?|contributions?|results?))",
    r"(?i)(conclusion[s]?|summary\s+of\s+(?:findings?|results?))",
    r"(?i)(our\s+(?:results?|findings?|contributions?))",
]

_LIMITATIONS_PATTERNS = [
    r"(?i)(limitation[s]?)",
    r"(?i)(drawback[s]?|weakness(?:es)?)",
    r"(?i)(constraint[s]?)",
    r"(?i)(we\s+do\s+not\s+address|beyond\s+the\s+scope)",
]

_FUTURE_WORK_PATTERNS = [
    r"(?i)(future\s+work|future\s+direction[s]?|future\s+research)",
    r"(?i)(open\s+problem[s]?|open\s+question[s]?)",
    r"(?i)(further\s+research|further\s+investigation)",
    r"(?i)(promising\s+(?:avenue[s]?|direction[s]?))",
]


class Summarizer:
    """
    Summarizes research papers and extracts structured insights.

    Usage:
        summ = Summarizer()
        summary   = summ.summarize(full_text)
        insights  = summ.extract_insights(full_text)
        analysis  = summ.full_analysis(chunks, source="paper.pdf")
    """

    def __init__(self, model_name: str = SUMMARIZATION_MODEL):
        logger.info(f"Loading summarization model: {model_name}")
        logger.info("  (First run downloads ~1.6 GB from HuggingFace — once only)")
        self.model_name = model_name
        self.tokenizer = None
        self.model = None

    def _load_model(self):
        """Lazy-initialise the summarization model."""
        if self.model is None:
            self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
            self.model = AutoModelForSeq2SeqLM.from_pretrained(self.model_name)
            logger.info("Summarization model loaded.")

    # ── Summarization ────────────────────────────────────────

    def summarize(
        self,
        text        : str,
        max_length  : int = 300,
        min_length  : int = 80,
        chunk_limit : int = 3000,   # chars to feed to BART (token limit ~1024)
    ) -> str:
        """
        Generate a concise summary of the provided text.

        BART has a token limit, so we truncate to chunk_limit
        characters before passing to the model.

        Args:
            text       : Full paper or section text.
            max_length : Max summary tokens.
            min_length : Min summary tokens.
            chunk_limit: Max input characters (BART token budget).

        Returns:
            A concise summary string.
        """
        if not text or not text.strip():
            return "No text provided for summarization."

        # Truncate to stay within model token limit
        input_text = text[:chunk_limit]

        logger.info(f"Summarizing {len(input_text):,} characters …")
        try:
            self._load_model()
            inputs = self.tokenizer([input_text], return_tensors="pt", max_length=1024, truncation=True)
            summary_ids = self.model.generate(
                inputs["input_ids"],
                max_length=max_length,
                min_length=min_length,
                early_stopping=True
            )
            summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            logger.info(f"Summary generated ({len(summary)} chars).")
            return summary
        except Exception as exc:
            logger.error(f"Summarization failed: {exc}")
            return f"Summarization error: {exc}"

    # ── Section extraction ───────────────────────────────────

    def _extract_section(self, text: str, patterns: list) -> list:
        """
        Extract sentences that follow a matched section header.

        Strategy:
          1. Find the first heading that matches any pattern.
          2. Extract up to 1000 characters.
          3. Split into sentences and discard any incomplete trailing sentence.

        Args:
            text    : Full paper text.
            patterns: List of regex patterns for section headers.

        Returns:
            List of sentences.
        """
        for pattern in patterns:
            match = re.search(pattern, text)
            if match:
                start = match.end() # Start AFTER the heading
                snippet = text[start : start + 1000]
                # Clean up newlines
                snippet = snippet.replace('\n', ' ')
                # Split into sentences
                sentences = re.split(r'(?<=[.!?])\s+', snippet)
                
                # Discard the last sentence if it's incomplete
                if sentences and not re.search(r'[.!?]$', sentences[-1].strip()):
                    sentences = sentences[:-1]
                
                # Filter out very short fragments
                cleaned = [s.strip() for s in sentences if len(s.strip()) > 20]
                if cleaned:
                    return cleaned

        return []

    def _extract_bullet_sentences(self, text: str, keywords: list) -> list:
        """
        Find sentences containing specific keywords and return as list.

        Fallback extraction when sections aren't explicitly labelled.
        """
        sentences = re.split(r'(?<=[.!?])\s+', text)
        hits = []
        for sent in sentences:
            sent_lower = sent.lower()
            if any(kw.lower() in sent_lower for kw in keywords):
                cleaned = sent.strip()
                if len(cleaned) > 20:  # skip very short fragments
                    hits.append(cleaned)
        return hits[:5]  # return top 5 matches

    # ── Insights extraction ──────────────────────────────────

    def extract_insights(self, text: str) -> dict:
        """
        Extract structured research insights from paper text.

        Returns:
            {
                "key_findings": str or list,
                "limitations" : str or list,
                "future_work" : str or list,
            }

        Strategy:
          - First, try to find explicit section headings.
          - If not found, fall back to sentence-level keyword search.
        """
        logger.info("Extracting research insights …")

        # ── Key Findings ──
        findings_text = self._extract_section(text, _FINDINGS_PATTERNS)
        if findings_text:
            key_findings = findings_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["we show", "we propose", "we demonstrate", "our method",
                 "achieves", "outperforms", "significantly", "novel", "key finding"]
            )
            key_findings = sentences if sentences else ["Not explicitly stated in the paper."]

        # ── Limitations ──
        limit_text = self._extract_section(text, _LIMITATIONS_PATTERNS)
        if limit_text:
            limitations = limit_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["limitation", "drawback", "weakness", "constraint",
                 "does not", "cannot", "unable to", "restricted to"]
            )
            limitations = sentences if sentences else ["Not explicitly stated in the paper."]

        # ── Future Work ──
        future_text = self._extract_section(text, _FUTURE_WORK_PATTERNS)
        if future_text:
            future_work = future_text
        else:
            sentences = self._extract_bullet_sentences(
                text,
                ["future work", "future research", "further investigation",
                 "open problem", "promising direction", "plan to", "will explore"]
            )
            future_work = sentences if sentences else ["Not explicitly stated in the paper."]

        logger.info("Insights extraction complete.")
        return {
            "key_findings": key_findings,
            "limitations" : limitations,
            "future_work" : future_work,
        }

    # ── Full analysis (convenience) ──────────────────────────

    def full_analysis(self, chunks: list, source: str = "paper") -> dict:
        """
        Run summary + insights on a set of chunks from one paper.

        Args:
            chunks: List of chunk dicts with 'text' and 'source'.
            source: Name of the paper for logging.

        Returns:
            {
                "source"      : str,
                "summary"     : str,
                "key_findings": str or list,
                "limitations" : str or list,
                "future_work" : str or list,
            }
        """
        logger.info(f"Full analysis for: {source}")

        # Reconstruct full text from chunks
        full_text = "\n\n".join(
            c["text"] for c in chunks
            if c.get("source") == source or source == "all"
        )

        if not full_text.strip():
            full_text = "\n\n".join(c["text"] for c in chunks)

        summary  = self.summarize(full_text)
        insights = self.extract_insights(full_text)

        return {
            "source"      : source,
            "summary"     : summary,
            **insights,
        }


# ── Quick test ───────────────────────────────────────────────
if __name__ == "__main__":
    sample_text = """
    Abstract
    We propose the Transformer, a new model architecture based entirely on attention
    mechanisms, dispensing with recurrence and convolutions. The model achieves state-
    of-the-art results on machine translation tasks.

    Key Findings
    Our model achieves 28.4 BLEU on WMT 2014 English-to-German translation, improving
    over existing best results by over 2 BLEU. We demonstrate that Transformers
    generalize well to other tasks by applying them to English constituency parsing.

    Limitations
    The model has a limitation in handling very long sequences due to the quadratic
    memory complexity of self-attention. We do not address streaming or online inference.

    Future Work
    Future work will explore sparse attention mechanisms to reduce computational cost.
    We plan to investigate applying Transformers to video and audio tasks.
    """

    summ = Summarizer()
    print("\n📝 Summary:")
    print(summ.summarize(sample_text))

    print("\n🔍 Insights:")
    insights = summ.extract_insights(sample_text)
    for key, val in insights.items():
        print(f"\n  {key.upper()}:")
        if isinstance(val, list):
            for item in val:
                print(f"    • {item}")
        else:
            print(f"    {val}")


In [ ]:
%%writefile app.py
# ============================================================
# app.py
# Phase 9 — Streamlit UI
#
# Research Paper Intelligence Engine
# A complete web interface for uploading PDFs, asking
# questions via RAG, generating summaries, and extracting
# structured research insights.
# ============================================================

import os
import sys
import tempfile
import logging

# Move HuggingFace cache to E: drive immediately because C: drive is full
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
os.environ["HF_HOME"] = os.path.join(BASE_DIR, "hf_cache")

import streamlit as st

# ── Path setup (so src/ is importable) ───────────────────────
BASE_DIR = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, BASE_DIR)

from config import TOP_K_RESULTS
from src.pdf_processor import PDFProcessor
from src.chunking      import TextChunker
from src.embeddings    import EmbeddingGenerator
from src.vector_db     import VectorDB
from src.retriever     import Retriever
from src.rag_pipeline  import RAGPipeline
from src.summarizer    import Summarizer

# ── Logging ──────────────────────────────────────────────────
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ============================================================
# Streamlit Page Configuration
# ============================================================
st.set_page_config(
    page_title  = "Research Paper Intelligence Engine",
    page_icon   = "🧠",
    layout      = "wide",
    initial_sidebar_state = "expanded",
)

# ============================================================
# Custom CSS — Premium Dark Theme
# ============================================================
st.markdown("""
<style>
/* ── Google Font ── */
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

/* ── Root Variables ── */
:root {
    --bg-primary   : #0d1117;
    --bg-secondary : #161b22;
    --bg-card      : #1c2128;
    --accent-blue  : #58a6ff;
    --accent-purple: #bc8cff;
    --accent-green : #3fb950;
    --accent-amber : #d29922;
    --text-primary : #e6edf3;
    --text-muted   : #8b949e;
    --border       : #30363d;
    --radius       : 12px;
}

/* ── Base ── */
html, body, [data-testid="stAppViewContainer"] {
    background-color: var(--bg-primary);
    font-family: 'Inter', sans-serif;
    color: var(--text-primary);
}

[data-testid="stSidebar"] {
    background-color: var(--bg-secondary);
    border-right: 1px solid var(--border);
}

/* ── Headers ── */
h1, h2, h3, h4 { font-family: 'Inter', sans-serif; font-weight: 600; }
h1 { font-size: 1.8rem; background: linear-gradient(135deg, var(--accent-blue), var(--accent-purple)); -webkit-background-clip: text; -webkit-text-fill-color: transparent; }
h2 { font-size: 1.3rem; color: var(--accent-blue); border-bottom: 1px solid var(--border); padding-bottom: 0.4rem; }
h3 { font-size: 1.1rem; color: var(--accent-purple); }

/* ── Cards ── */
.rag-card {
    background    : var(--bg-card);
    border        : 1px solid var(--border);
    border-radius : var(--radius);
    padding       : 1.2rem 1.5rem;
    margin-bottom : 1rem;
    transition    : border-color 0.2s ease;
}
.rag-card:hover { border-color: var(--accent-blue); }

/* ── Answer box ── */
.answer-box {
    background    : linear-gradient(135deg, #1a2744, #1c2128);
    border        : 1px solid var(--accent-blue);
    border-radius : var(--radius);
    padding       : 1.2rem 1.5rem;
    font-size     : 1.05rem;
    line-height   : 1.7;
    margin-bottom : 1rem;
}

/* ── Insight sections ── */
.insight-findings  { border-left: 4px solid var(--accent-green);  background: #0d2010; }
.insight-limits    { border-left: 4px solid var(--accent-amber);  background: #1f1700; }
.insight-future    { border-left: 4px solid var(--accent-purple); background: #1a0d2e; }
.insight-box {
    border-radius : var(--radius);
    padding       : 1rem 1.2rem;
    margin-bottom : 0.8rem;
    font-size     : 0.95rem;
    line-height   : 1.6;
}

/* ── Confidence badge ── */
.badge {
    display       : inline-block;
    padding       : 0.2rem 0.7rem;
    border-radius : 20px;
    font-size     : 0.78rem;
    font-weight   : 600;
    margin-left   : 0.5rem;
}
.badge-high   { background: #1a3a1a; color: var(--accent-green); border: 1px solid var(--accent-green); }
.badge-medium { background: #2a2000; color: var(--accent-amber); border: 1px solid var(--accent-amber); }
.badge-low    { background: #2a0000; color: #ff7b72;             border: 1px solid #ff7b72; }

/* ── Source chips ── */
.source-chip {
    display       : inline-block;
    background    : #21262d;
    border        : 1px solid var(--border);
    border-radius : 6px;
    padding       : 0.1rem 0.6rem;
    font-size     : 0.75rem;
    color         : var(--text-muted);
    margin        : 0.2rem 0.2rem 0.2rem 0;
    font-family   : 'JetBrains Mono', monospace;
}

/* ── Buttons ── */
[data-testid="stButton"] > button {
    background    : linear-gradient(135deg, #1f6feb, #388bfd);
    color         : white;
    border        : none;
    border-radius : 8px;
    font-weight   : 500;
    padding       : 0.5rem 1.2rem;
    transition    : opacity 0.2s ease, transform 0.1s ease;
}
[data-testid="stButton"] > button:hover {
    opacity   : 0.88;
    transform : translateY(-1px);
}
[data-testid="stButton"] > button:active { transform: translateY(0); }

/* ── File uploader ── */
[data-testid="stFileUploader"] {
    border        : 1.5px dashed var(--border);
    border-radius : var(--radius);
    background    : var(--bg-card);
    transition    : border-color 0.2s;
}
[data-testid="stFileUploader"]:hover { border-color: var(--accent-blue); }

/* ── Text input / textarea ── */
[data-testid="stTextInput"] input,
[data-testid="stTextArea"] textarea {
    background    : var(--bg-card);
    border        : 1px solid var(--border);
    border-radius : 8px;
    color         : var(--text-primary);
    font-family   : 'Inter', sans-serif;
}

/* ── Progress bar ── */
[data-testid="stProgress"] > div > div { background: var(--accent-blue); }

/* ── Expander ── */
[data-testid="stExpander"] { border: 1px solid var(--border); border-radius: var(--radius); }

/* ── Divider ── */
hr { border-color: var(--border); }

/* ── Metric ── */
[data-testid="stMetric"] { background: var(--bg-card); border: 1px solid var(--border); border-radius: var(--radius); padding: 0.8rem; }
</style>
""", unsafe_allow_html=True)


# ============================================================
# Session State Initialisation
# ============================================================
def init_session():
    defaults = {
        "index_built"   : False,
        "chunks"        : [],
        "sources"       : [],
        "rag_pipeline"  : None,
        "summarizer"    : None,
        "retriever"     : None,
        "embedding_gen" : None,
        "vector_db"     : None,
    }
    for key, val in defaults.items():
        if key not in st.session_state:
            st.session_state[key] = val

init_session()


# ============================================================
# Cached Resource Loaders (load once per session)
# ============================================================
@st.cache_resource(show_spinner="Loading embedding model…")
def load_embedding_gen():
    return EmbeddingGenerator()

@st.cache_resource(show_spinner="Loading QA model…")
def load_rag_pipeline(retriever):
    return RAGPipeline(retriever=retriever)

@st.cache_resource(show_spinner="Loading summarizer…")
def load_summarizer():
    return Summarizer()


# ============================================================
# Helper: Confidence Badge HTML
# ============================================================
def confidence_badge(score: float) -> str:
    return '<span class="badge badge-high" style="background: #1a1a3a; color: var(--accent-blue); border: 1px solid var(--accent-blue);">🤖 Synthesized</span>'


# ============================================================
# Helper: Format Insight Block
# ============================================================
def render_insight(title: str, content, css_class: str, icon: str):
    if isinstance(content, list):
        items_html = "".join(f"<li>{item}</li>" for item in content if item)
        body = f"<ul>{items_html}</ul>"
    else:
        body = f"<p>{content}</p>"

    st.markdown(f"""
    <div class="insight-box {css_class}">
        <strong>{icon} {title}</strong><br/>
        {body}
    </div>
    """, unsafe_allow_html=True)


# ============================================================
# SIDEBAR
# ============================================================
with st.sidebar:
    st.markdown("## 🧠 Research Engine")
    st.markdown("*Powered by RAG + HuggingFace*")
    st.divider()

    # ── Upload Section ──
    st.markdown("### 📂 Upload Papers")
    uploaded_files = st.file_uploader(
        label       = "Drop PDF files here",
        type        = ["pdf"],
        accept_multiple_files = True,
        key         = "pdf_uploader",
        help        = "Upload one or more research paper PDFs.",
    )

    # ── Chunking settings ──
    st.divider()
    st.markdown("### ⚙️ Settings")
    chunk_size    = st.slider("Chunk Size (chars)",    200, 1000, 500, 50,
                              help="Size of each text chunk passed to the embedder.")
    chunk_overlap = st.slider("Chunk Overlap (chars)",  0,  200, 100, 10,
                              help="Overlap between consecutive chunks.")
    top_k         = st.slider("Top-K Retrieval",        1,   10,   5,  1,
                              help="Number of context chunks retrieved per query.")

    # ── Process button ──
    st.divider()
    process_btn = st.button("🚀 Process & Index PDFs", use_container_width=True)

    # ── Index stats ──
    if st.session_state.index_built:
        st.divider()
        st.markdown("### 📊 Index Stats")
        vdb   = st.session_state.vector_db
        stats = vdb.get_stats() if vdb else {}
        st.metric("Total Chunks",   stats.get("total_chunks",  0))
        st.metric("Total Vectors",  stats.get("total_vectors", 0))
        sources = stats.get("sources", [])
        st.markdown(f"**Papers indexed:** {len(sources)}")
        for s in sources:
            st.markdown(f'<span class="source-chip">📄 {s}</span>', unsafe_allow_html=True)

    # ── Footer ──
    st.divider()
    st.caption("Phase 1 of Agentic AI Research Scientist System")
    st.caption("Built with ❤️ using Streamlit + LangChain + FAISS")


# ============================================================
# MAIN — Header
# ============================================================
st.markdown("# 🧠 Research Paper Intelligence Engine")
st.markdown(
    "Upload research papers → ask questions → get AI-powered answers, "
    "summaries, and structured insights."
)
st.divider()


# ============================================================
# PDF Processing Logic (triggered by button)
# ============================================================
if process_btn:
    if not uploaded_files:
        st.warning("⚠️ Please upload at least one PDF before processing.")
    else:
        with st.status("📥 Processing PDFs…", expanded=True) as status:
            try:
                processor   = PDFProcessor()
                chunker     = TextChunker(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
                embed_gen   = load_embedding_gen()
                vdb         = VectorDB()

                all_chunks  = []
                docs        = {}

                # Step 1: Extract text from each PDF
                for i, uf in enumerate(uploaded_files):
                    st.write(f"📄 Extracting: **{uf.name}**")

                    # Step 1: Extract text directly from memory buffer
                    result = processor.process_single(stream=uf.getvalue(), filename=uf.name)
                    docs[uf.name] = result

                # Step 2: Chunk all documents
                st.write("✂️ Chunking text…")
                all_chunks = chunker.chunk_documents(docs)

                # Fix source names (temp file names → original PDF names)
                # Already handled because we pass docs with original keys

                # Step 3: Generate embeddings
                st.write(f"🔢 Generating embeddings for {len(all_chunks)} chunks…")
                embeddings = embed_gen.embed_chunks(all_chunks)

                # Step 4: Build FAISS index
                st.write("🗂️ Building FAISS index…")
                vdb.build_index(embeddings, all_chunks)
                vdb.save()

                # Save to session state
                retriever = Retriever(embedding_gen=embed_gen, vector_db=vdb)
                st.session_state.index_built   = True
                st.session_state.chunks        = all_chunks
                st.session_state.sources       = list(docs.keys())
                st.session_state.vector_db     = vdb
                st.session_state.embedding_gen = embed_gen
                st.session_state.retriever     = retriever

                status.update(
                    label=f"✅ Indexed {len(all_chunks)} chunks from {len(docs)} PDF(s)!",
                    state="complete",
                )
            except Exception as exc:
                status.update(label=f"❌ Error: {exc}", state="error")
                logger.exception(exc)


# ============================================================
# TABS: Q&A | Summarize | Insights | Retrieved Context
# ============================================================
tab_qa, tab_summary, tab_insights, tab_context = st.tabs([
    "💬 Ask a Question",
    "📝 Summarize Paper",
    "🔍 Research Insights",
    "📚 Retrieved Context",
])


# ──────────────────────────────────────────────────────────
# TAB 1: Q&A via RAG
# ──────────────────────────────────────────────────────────
with tab_qa:
    st.markdown("## 💬 Ask Questions About Your Papers")
    st.markdown("*Ask anything — the AI retrieves relevant context and answers from your papers.*")

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first using the sidebar.")
    else:
        question = st.text_input(
            label       = "Your question",
            placeholder = "What is the main contribution of this paper?",
            key         = "qa_question",
        )

        col1, col2 = st.columns([1, 5])
        with col1:
            ask_btn = st.button("🔎 Ask", key="ask_btn", use_container_width=True)

        if ask_btn and question:
            with st.spinner("🤔 Thinking…"):
                try:
                    rag = RAGPipeline(
                        retriever=st.session_state.retriever,
                    )
                    result = rag.answer(question, top_k=top_k)

                    # ── Answer box ──
                    badge = confidence_badge(result["confidence"])
                    st.markdown(f"### Answer {badge}", unsafe_allow_html=True)
                    st.markdown(
                        f'<div class="answer-box">{result["answer"]}</div>',
                        unsafe_allow_html=True,
                    )

                    # ── Sources ──
                    st.markdown("**Sources used:**")
                    unique_sources = {s["source"] for s in result["sources"]}
                    chips = " ".join(
                        f'<span class="source-chip">📄 {s}</span>'
                        for s in unique_sources
                    )
                    st.markdown(chips, unsafe_allow_html=True)

                    # ── Expandable context ──
                    with st.expander("🔎 View retrieved context"):
                        st.code(result["context"][:2000], language=None)

                except Exception as exc:
                    st.error(f"❌ Error: {exc}")
                    logger.exception(exc)

        elif ask_btn and not question:
            st.warning("Please type a question first.")

        # ── Example questions ──
        st.divider()
        st.markdown("**💡 Example questions:**")
        examples = [
            "What is the main contribution of this paper?",
            "What dataset was used for evaluation?",
            "What are the experimental results?",
            "What deep learning architecture is proposed?",
            "What problem does this paper solve?",
        ]
        
        def set_q(q):
            st.session_state.qa_question = q

        for ex in examples:
            st.button(f"▷ {ex}", key=f"ex_{ex[:20]}", on_click=set_q, args=(ex,))


# ──────────────────────────────────────────────────────────
# TAB 2: Summarization
# ──────────────────────────────────────────────────────────
with tab_summary:
    st.markdown("## 📝 Paper Summarization")
    st.markdown("*Generate a concise abstract-style summary of any indexed paper.*")

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        sources = st.session_state.sources
        selected = st.selectbox(
            "Select a paper to summarize",
            options=["All Papers"] + sources,
            key="summary_source",
        )

        sum_btn = st.button("📝 Generate Summary", key="sum_btn")

        if sum_btn:
            with st.spinner("🤔 Generating summary with BART…"):
                try:
                    summ = load_summarizer()
                    chunks = st.session_state.chunks

                    if selected == "All Papers":
                        full_text = "\n\n".join(c["text"] for c in chunks)
                        src_label = "All Papers"
                    else:
                        full_text = "\n\n".join(
                            c["text"] for c in chunks if c.get("source") == selected
                        )
                        src_label = selected

                    summary = summ.summarize(full_text)

                    st.markdown(f"### Summary: *{src_label}*")
                    st.markdown(
                        f'<div class="rag-card">{summary}</div>',
                        unsafe_allow_html=True,
                    )
                    st.download_button(
                        "⬇️ Download Summary",
                        data=summary,
                        file_name=f"summary_{src_label.replace(' ','_')}.txt",
                        mime="text/plain",
                        key="download_summary",
                    )
                except Exception as exc:
                    st.error(f"❌ Error: {exc}")
                    logger.exception(exc)


# ──────────────────────────────────────────────────────────
# TAB 3: Research Insights
# ──────────────────────────────────────────────────────────
with tab_insights:
    st.markdown("## 🔍 Research Insights Extraction")
    st.markdown(
        "*Automatically extract **Key Findings**, **Limitations**, and **Future Work** "
        "from your research paper.*"
    )

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        sources  = st.session_state.sources
        selected = st.selectbox(
            "Select a paper",
            options=sources,
            key="insights_source",
        )

        ins_btn = st.button("🔍 Extract Insights", key="ins_btn")

        if ins_btn:
            with st.spinner("🕵️ Extracting structured insights…"):
                try:
                    summ   = load_summarizer()
                    chunks = st.session_state.chunks
                    result = summ.full_analysis(chunks, source=selected)

                    # ── Summary card ──
                    st.markdown(f"### 📄 {selected}")
                    st.markdown(
                        f'<div class="rag-card"><strong>Summary:</strong> {result["summary"]}</div>',
                        unsafe_allow_html=True,
                    )

                    # ── Insight cards ──
                    col1, col2, col3 = st.columns(3)

                    with col1:
                        render_insight(
                            "Key Findings",
                            result["key_findings"],
                            "insight-box insight-findings",
                            "🟢",
                        )
                    with col2:
                        render_insight(
                            "Limitations",
                            result["limitations"],
                            "insight-box insight-limits",
                            "🟡",
                        )
                    with col3:
                        render_insight(
                            "Future Work",
                            result["future_work"],
                            "insight-box insight-future",
                            "🔵",
                        )

                    # ── Download ──
                    import json
                    report = json.dumps(result, indent=2, ensure_ascii=False)
                    st.download_button(
                        "⬇️ Download Insights (JSON)",
                        data=report,
                        file_name=f"insights_{selected.replace(' ','_')}.json",
                        mime="application/json",
                        key="download_insights",
                    )

                except Exception as exc:
                    st.error(f"❌ Error: {exc}")
                    logger.exception(exc)


# ──────────────────────────────────────────────────────────
# TAB 4: Retrieved Context Explorer
# ──────────────────────────────────────────────────────────
with tab_context:
    st.markdown("## 📚 Semantic Search Explorer")
    st.markdown(
        "*Search the indexed chunks directly to see exactly what chunks are retrieved "
        "for any query — useful for debugging and understanding the pipeline.*"
    )

    if not st.session_state.index_built:
        st.info("👈 Upload and process PDFs first.")
    else:
        search_query = st.text_input(
            "Search query",
            placeholder="attention mechanism transformer",
            key="ctx_query",
        )
        search_btn = st.button("🔎 Search Chunks", key="ctx_btn")

        if search_btn and search_query:
            with st.spinner("Searching…"):
                try:
                    retriever = st.session_state.retriever
                    results   = retriever.retrieve(search_query, top_k=top_k)

                    st.markdown(f"**Found {len(results)} chunks:**")
                    for r in results:
                        with st.expander(
                            f"Rank {r['rank']} | Score: {r['score']:.4f} | 📄 {r['source']}"
                        ):
                            st.text(r["text"])

                except Exception as exc:
                    st.error(f"❌ Error: {exc}")


### Install Dependencies

In [ ]:
!pip install -r requirements.txt
!npm install localtunnel

### Get your Endpoint IP
Copy the IP address below. You will need to paste it into the localtunnel website.

In [ ]:
import urllib
print("Password/Endpoint IP for localtunnel is:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())

### Launch App
Click the `loca.lt` link below and paste the IP address!

In [ ]:
!streamlit run app.py &>/content/logs.txt & npx localtunnel --port 8501